In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sabahesaraki/breast-ultrasound-images-dataset")

print("Path to dataset files:", path)

data_dir = f"{path}/Dataset_BUSI_with_GT"

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    DepthwiseConv2D, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Activation
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

import tensorflow as tf


In [ ]:
classes = ['benign','malignant','normal']

images = []
labels = []

for label, cls in enumerate(classes):
    path = os.path.join(data_dir, cls)

    for img in os.listdir(path):

        # Ignore ground truth mask images
        if "mask" in img.lower():
            continue

        img_path = os.path.join(path, img)

        image = cv2.imread(img_path)
        image = cv2.resize(image, (128,128))

        images.append(image)
        labels.append(label)

images = np.array(images) / 255.0
labels = np.array(labels)

print("Dataset size:", images.shape)

In [ ]:
labels_cat = to_categorical(labels,3)

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    images,
    labels_cat,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

In [ ]:
def create_model():
    """
    CNN using Depthwise Separable Convolutions instead of standard Conv2D.
    Each DSC block = DepthwiseConv2D (spatial filtering) + 1x1 Conv2D (channel mixing).
    This significantly reduces parameters while maintaining representational power.
    """
    model = Sequential()

    # --- Block 1: 32 filters ---
    model.add(DepthwiseConv2D((3,3), padding='same', use_bias=False, input_shape=(128,128,3)))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Conv2D(32, (1,1), padding='same', use_bias=False))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(2, 2))

    # --- Block 2: 64 filters ---
    model.add(DepthwiseConv2D((3,3), padding='same', use_bias=False))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Conv2D(64, (1,1), padding='same', use_bias=False))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(2, 2))

    # --- Block 3: 128 filters ---
    model.add(DepthwiseConv2D((3,3), padding='same', use_bias=False))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Conv2D(128, (1,1), padding='same', use_bias=False))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(2, 2))

    model.add(Flatten())

    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))

    model.add(Dense(3, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:
model = create_model()

history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.1
)

In [ ]:
def evaluate_model(model,X_test,y_test):

    pred = model.predict(X_test)
    pred_labels = np.argmax(pred,axis=1)
    true_labels = np.argmax(y_test,axis=1)

    acc = accuracy_score(true_labels,pred_labels)
    prec = precision_score(true_labels,pred_labels,average='weighted')
    rec = recall_score(true_labels,pred_labels,average='weighted')
    f1 = f1_score(true_labels,pred_labels,average='weighted')

    return acc,prec,rec,f1

In [ ]:
base_results = evaluate_model(model,X_test,y_test)
print("Base CNN:",base_results)

In [ ]:
y_labels = np.argmax(y_train,axis=1)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_labels),
    y=y_labels
)

class_weights = dict(enumerate(class_weights))

In [ ]:
model_weights = create_model()

model_weights.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    class_weight=class_weights
)

weights_results = evaluate_model(model_weights,X_test,y_test)

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

datagen.fit(X_train)

model_aug = create_model()

model_aug.fit(
    datagen.flow(X_train,y_train,batch_size=32),
    epochs=15
)

aug_results = evaluate_model(model_aug,X_test,y_test)

In [ ]:
def focal_loss(gamma=2., alpha=.25):

    def loss(y_true,y_pred):

        epsilon = 1e-7
        y_pred = tf.clip_by_value(y_pred,epsilon,1-epsilon)

        cross_entropy = -y_true*tf.math.log(y_pred)
        weight = alpha*tf.math.pow(1-y_pred,gamma)

        return tf.reduce_sum(weight*cross_entropy,axis=1)

    return loss

In [ ]:
model_focal = create_model()

model_focal.compile(
    optimizer='adam',
    loss=focal_loss(),
    metrics=['accuracy']
)

model_focal.fit(
    X_train,
    y_train,
    epochs=15
)

focal_results = evaluate_model(model_focal,X_test,y_test)

In [ ]:
from imblearn.over_sampling import SMOTE

X_flat = X_train.reshape(len(X_train),-1)
y_labels = np.argmax(y_train,axis=1)

smote = SMOTE()

X_res,y_res = smote.fit_resample(X_flat,y_labels)

X_res = X_res.reshape(-1,128,128,3)
y_res = to_categorical(y_res)

In [ ]:
model_smote = create_model()

model_smote.fit(
    X_res,
    y_res,
    epochs=15
)

smote_results = evaluate_model(model_smote,X_test,y_test)

In [ ]:
results = pd.DataFrame({
    "Model":[
        "Base CNN",
        "Class Weights",
        "Data Augmentation",
        "Focal Loss",
        "SMOTE"
    ],
    "Accuracy":[
        base_results[0],
        weights_results[0],
        aug_results[0],
        focal_results[0],
        smote_results[0]
    ],
    "Precision":[
        base_results[1],
        weights_results[1],
        aug_results[1],
        focal_results[1],
        smote_results[1]
    ],
    "Recall":[
        base_results[2],
        weights_results[2],
        aug_results[2],
        focal_results[2],
        smote_results[2]
    ],
    "F1 Score":[
        base_results[3],
        weights_results[3],
        aug_results[3],
        focal_results[3],
        smote_results[3]
    ]
})

print(results)

In [ ]:
plt.figure()
plt.bar(results["Model"],results["Accuracy"])
plt.xticks(rotation=45)
plt.title("Accuracy Comparison")
plt.show()

In [ ]:
plt.figure()
plt.bar(results["Model"],results["F1 Score"])
plt.xticks(rotation=45)
plt.title("F1 Score Comparison")
plt.show()

In [ ]:
plt.figure()
plt.plot(results["Model"],results["Precision"],marker='o')
plt.plot(results["Model"],results["Recall"],marker='o')
plt.legend(["Precision","Recall"])
plt.xticks(rotation=45)
plt.title("Precision vs Recall")
plt.show()